# 06 - Seleção de atributos e modelos clássicos (módulos 8 e 9)

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from harmonic_signatures.config import PROCESSED_DIR
from harmonic_signatures.features.selection import select_features
from harmonic_signatures.models.classical import compare_models

harm_matrix = pd.read_parquet(PROCESSED_DIR / 'combined_matrix.parquet')
feature_cols = [c for c in harm_matrix.columns if c.startswith('H') and (c.endswith('_avg') or c.endswith('_max')) and '_harm' not in c and '_inter' not in c]
print('candidatas:', len(feature_cols))

candidatas: 300


## Seleção automática (ANOVA F-score + remoção de redundância por correlação)

In [2]:
result = select_features(harm_matrix, feature_cols, method='f_classif', corr_threshold=0.95, top_k=40)
print('selecionadas:', len(result.selected_features), '| descartadas por redundância:', len(result.dropped_redundant))
result.ranking.head(10)

selecionadas: 40 | descartadas por redundância: 119


,feature,score,p_value
0,H24_Ic_avg,1.488770e+06,0.0
1,H18_Ic_avg,1.315910e+06,0.0
2,H12_Ic_avg,1.209843e+06,0.0
3,H06_Ic_avg,1.150938e+06,0.0
4,H23_Ib_avg,1.083654e+06,0.0
5,H30_Ic_avg,1.045044e+06,0.0
6,H24_Ib_avg,9.900257e+05,0.0
7,H12_Ib_avg,9.848059e+05,0.0
8,H02_Ib_avg,9.716977e+05,0.0
9,H04_Ib_avg,9.110812e+05,0.0


In [3]:
result.summarize_by('variable', 'statistic')

,variable,statistic,n_selected
0,I,avg,29
1,V,avg,11


## Comparação automática de modelos clássicos (RF, XGBoost, SVM, MLP, kNN, Regressão Logística)

In [4]:
X = harm_matrix[result.selected_features].fillna(0).to_numpy()
y = harm_matrix['label'].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
summary, results = compare_models(X_train, y_train, X_test, y_test)
summary

,model,accuracy,f1_macro,fit_seconds,predict_seconds
0,svm,0.922919,0.938939,0.663223,1.587627
1,logistic_regression,0.920868,0.937325,0.806815,0.001000
2,mlp,0.919672,0.925521,3.302046,0.003875
3,knn,0.891301,0.913993,0.006003,1.517098
4,random_forest,0.873184,0.899662,0.984755,0.090722
5,xgboost,0.872671,0.899327,2.596498,0.030836
